# 🍐 Little Fig Studio

**Train any LLM on any hardware.** Select your model, configure, launch.

| Feature | Result |
|---|---|
| Quantization | Beats NF4 on 156/156 layers |
| GPU Speed | 7× faster than BnB NF4 |
| CPU Training | 1.1B model in 400MB RAM |
| Optimizer | FigMeZO (−18.6%, original research) |

**Run all cells below ↓**

In [ ]:
# Step 1: Install Little Fig + dependencies
!git clone https://github.com/ticketguy/littlefig.git /content/littlefig 2>/dev/null || (cd /content/littlefig && git pull)

# Install with [train] extras (embers-diaries is not yet on PyPI)
!pip install -e "/content/littlefig[train]" -q

# Install cloudflared binary for tunnel
import subprocess, os
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/usr/local/bin/cloudflared'], check=True)
    os.chmod('/usr/local/bin/cloudflared', 0o755)
    print('✅ cloudflared installed')
else:
    print('✅ cloudflared already installed')

# Verify install
import sys, importlib
# Add source path and clear any cached import failures from prior runs
if '/content/littlefig/src' not in sys.path:
    sys.path.insert(0, '/content/littlefig/src')
# Remove any cached failed imports of little_fig
for key in list(sys.modules.keys()):
    if 'little_fig' in key:
        del sys.modules[key]
importlib.invalidate_caches()

import torch
from little_fig.engine import __version__ as fig_version
print(f'✅ Little Fig v{fig_version}')
print(f'✅ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 2: Launch Little Fig Studio
import subprocess, time, re, sys
sys.path.insert(0, '/content/littlefig/src')

# Start the web server
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'little_fig.web.server:app',
     '--host', '0.0.0.0', '--port', '8888'],
    cwd='/content/littlefig/src',
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)

# Check if server started
import urllib.request
server_ok = False
try:
    urllib.request.urlopen('http://localhost:8888/api/health', timeout=3)
    server_ok = True
    print('✅ Server running on port 8888')
except Exception as e:
    # Server might return non-200 but still be alive on /
    try:
        urllib.request.urlopen('http://localhost:8888', timeout=3)
        server_ok = True
        print('✅ Server running on port 8888')
    except urllib.error.HTTPError:
        # HTTP error means server IS running (just returned non-200)
        server_ok = True
        print('✅ Server running on port 8888')
    except Exception:
        print('⚠️ Server may not have started. Checking stderr...')
        try:
            err = server.stderr.read(2000).decode()
            if err:
                print(err[:800])
        except:
            pass

if not server_ok:
    print('\n❌ Server failed to start. Check the error above.')
    print('   Common fixes:')
    print('   - Re-run Step 1 (install cell)')
    print('   - Runtime > Restart runtime, then re-run all cells')
else:
    # Start cloudflared tunnel (free, no account needed)
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8888'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(6)

    # Read tunnel URL from stderr (cloudflared prints there)
    tunnel_url = None
    try:
        import select
        # Read available stderr
        output = b''
        for _ in range(10):
            chunk = tunnel.stderr.read1(4096) if hasattr(tunnel.stderr, 'read1') else b''
            if not chunk:
                # Try non-blocking read
                import os, fcntl
                fd = tunnel.stderr.fileno()
                fl = fcntl.fcntl(fd, fcntl.F_GETFL)
                fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)
                try:
                    chunk = tunnel.stderr.read(8192)
                except:
                    chunk = b''
                finally:
                    fcntl.fcntl(fd, fcntl.F_SETFL, fl)
            if chunk:
                output += chunk
            else:
                time.sleep(0.5)
        text = output.decode(errors='ignore')
        urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if urls:
            tunnel_url = urls[0]
    except Exception as e:
        pass

    # Fallback: try reading after more time
    if not tunnel_url:
        time.sleep(4)
        try:
            import os, fcntl
            fd = tunnel.stderr.fileno()
            fl = fcntl.fcntl(fd, fcntl.F_GETFL)
            fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)
            try:
                extra = tunnel.stderr.read(8192)
                if extra:
                    text = extra.decode(errors='ignore')
                    urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
                    if urls:
                        tunnel_url = urls[0]
            except:
                pass
            finally:
                fcntl.fcntl(fd, fcntl.F_SETFL, fl)
        except:
            pass

    if tunnel_url:
        print(f'\n\n🍐 Little Fig Studio is LIVE!')
        print(f'\n   👉 {tunnel_url}')
        print(f'\n   Open the link above in your browser.')
        print(f'   Select model → Configure → Train')
        print(f'\n   Keep this cell running to keep the server alive.')
    else:
        print('\n⚠️ Cloudflare tunnel could not extract URL.')
        print('   Falling back to Colab proxy...')
        try:
            from google.colab.output import eval_js
            from IPython.display import display, HTML
            proxy_url = eval_js("google.colab.kernel.proxyPort(8888)")
            print(f'\n🍐 Little Fig Studio:')
            print(f'   👉 {proxy_url}')
            display(HTML(f'<a href="{proxy_url}" target="_blank" style="font-size:16px;">Open Little Fig Studio →</a>'))
        except Exception as e:
            print(f'\n   Colab proxy also failed: {e}')
            print(f'   The server IS running on port 8888.')
            print(f'   Try: Runtime > Restart, then re-run all cells.')

---
## Alternative: Python API (no UI)

Change `MODEL` to any HuggingFace model.

In [ ]:
import sys
sys.path.insert(0, '/content/littlefig/src')
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig

# === YOUR MODEL HERE ===
MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# MODEL = 'google/gemma-3-4b-it'
# MODEL = 'Qwen/Qwen2.5-1.5B'
# MODEL = 'microsoft/phi-2'

model = FigModel.from_pretrained(MODEL, lora_r=16, lora_alpha=32, shared_codebook=True)
print(f'✅ {MODEL} loaded | Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Train
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)
trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)
trainer.train()
model.save_adapter('./my_adapter')
print('\n✅ Training complete! Adapter saved.')

---
*0xticketguy / Harboria Labs | AGPL-3.0*